# Similarity text search with NewsGroups - student

Big picture:  
	1.	Load 20 Newsgroups from Hugging Face (SetFit/20_newsgroups).  
	2.	Chunk each post with chunk_text() or chunk_text_chars().  
	3.	Embeds chunks with sentence-transformers/all-MiniLM-L6-v2.  
	4.	Upserts vectors + payload into Qdrant.  
	5.	Queries Qdrant to retrieve similar chunks.  

Run in qdrant_mps


In [ ]:
# installs 
# !pip install -U qdrant-client sentence-transformers datasets numpy


In [ ]:
#imports
import re
import time

from typing import Dict, Any, List
import numpy as np

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.models import Filter, FieldCondition, MatchValue

from datasets import load_dataset

# TODO: comment re.compile(...) and regexp
_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

print("ok")

In [ ]:
def wc(s: str) -> int:
    return s.count(" ") + 1


def chunk_text(
    text: str,
    max_words: int = 120,
    overlap_words: int = 30,
    min_words: int = 30,
) -> List[str]:
    """
    ToDo: Comment the function operation
    """
    
    if overlap_words >= max_words:
        overlap_words = max_words // 4  # or just set to 0

    MAX_CHARS_TOTAL = 20000     # or 30000
    MAX_SENTS = 500             # prevents pathological split explosion

    # TODO: Comment this
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    
    if len(text) > MAX_CHARS_TOTAL:
        text = text[:MAX_CHARS_TOTAL]

    sents = _SENT_SPLIT_RE.split(text)
    if len(sents) > MAX_SENTS:
        # fall back to char chunking or just truncate the sentence list
        sents = sents[:MAX_SENTS]

    sents = [s.strip() for s in sents if s.strip()]

    chunks = []
    cur = []
    cur_words = 0

    for s in sents:
        w = wc(s)

        # If a single sentence is huge, hard-split it
        if w > max_words:
            if cur_words >= min_words:
                chunks.append(" ".join(cur).strip())
            cur, cur_words = [], 0

            tokens = s.split()
            start = 0
            prev_start = -1

            while start < len(tokens):
                end = min(start + max_words, len(tokens))
                part = " ".join(tokens[start:end]).strip()
                if wc(part) >= min_words:
                    chunks.append(part)

                # compute next start with overlap
                next_start = end - overlap_words if overlap_words > 0 else end

                # GUARANTEE progress (prevents infinite loops)
                if next_start <= start:
                    next_start = end  # drop overlap if it would stall

                prev_start, start = start, next_start
                # start = end - overlap_words if overlap_words > 0 else end
            continue

        # flush if adding would exceed max_words
        if cur and (cur_words + w > max_words):
            chunk = " ".join(cur).strip()
            if wc(chunk) >= min_words:
                chunks.append(chunk)

            if overlap_words > 0:
                tail = chunk.split()[-overlap_words:]
                cur = [" ".join(tail)]
                cur_words = len(tail)
            else:
                cur, cur_words = [], 0

        cur.append(s)
        cur_words += w

    if cur:
        chunk = " ".join(cur).strip()
        if wc(chunk) >= min_words:
            chunks.append(chunk)

    return chunks

In [ ]:
# Better chunk_text based on chars (no punctuation chars)
# Some ng texts might be tricky with punctuations
def normalize_ws(text: str) -> str:
    # keep newlines to help paragraph-based splitting
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # collapse excessive spaces but keep newlines
    text = re.sub(r"[ \t]+", " ", text)
    # collapse too many blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def chunk_text_chars(
    text: str,
    chunk_chars: int = 800,        # ~ 120-160 words depending on language/style
    overlap_chars: int = 100,      # 200 original
    min_chars: int = 200,
    max_chars_total: int = 20000,  # hard cap per document
    max_chunks: int = 6,           # hard cap per document
) -> List[str]:
    """
    ToDo: Comment the function operation
    """
    if not text:
        return []

    text = normalize_ws(text)
    if not text:
        return []

    if len(text) > max_chars_total:
        text = text[:max_chars_total]

    # Prefer paragraph boundaries (cheap). If no paragraphs, fallback to raw text.
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    if not paras:
        paras = [text]

    chunks: List[str] = []
    buf = ""

    def flush_buf(force=False):
        nonlocal buf
        s = buf.strip()
        if len(s) >= min_chars or force:
            if s:
                chunks.append(s)
        buf = ""

    for p in paras:
        if len(chunks) >= max_chunks:
            break

        # If paragraph itself is huge, slice it directly into windows
        if len(p) > chunk_chars:
            flush_buf()
            start = 0
            while start < len(p) and len(chunks) < max_chunks:
                end = min(start + chunk_chars, len(p))
                piece = p[start:end].strip()
                if len(piece) >= min_chars:
                    chunks.append(piece)
                start = end - overlap_chars if overlap_chars > 0 else end
            continue

        # Try to pack paragraphs into buffer up to chunk_chars
        if not buf:
            buf = p
        elif len(buf) + 2 + len(p) <= chunk_chars:
            buf = buf + "\n\n" + p
        else:
            flush_buf()
            if len(chunks) >= max_chunks:
                break
            buf = p

    if len(chunks) < max_chunks and buf:
        flush_buf(force=True)

    # Add overlap between adjacent chunks (character overlap window)
    if overlap_chars > 0 and len(chunks) > 1:
        overlapped = []
        prev_tail = ""
        for c in chunks:
            if prev_tail:
                c2 = (prev_tail + "\n" + c).strip()
            else:
                c2 = c
            overlapped.append(c2)
            prev_tail = c[-overlap_chars:]
        chunks = overlapped[:max_chunks]

    return chunks

In [ ]:
def chunk_text_hybrid(
    text: str,
    max_words: int = 120,
    overlap_words: int = 30,
    min_words: int = 30,
    max_chunks: int = 6,
) -> List[str]:
    """
    ToDo: Comment the function operation
    """
    t = text or ""
    t_len = len(t)
    punct = sum(t.count(x) for x in ".!?")

    # Heuristic: long text with very few sentence markers => use char chunking
    if t_len > 12000 and punct < 10:
        print(f"--using chunk_text_chars len {t_len} punct {punct}")
        return chunk_text_chars(
            t,
            chunk_chars=800,
            overlap_chars=200,
            min_chars=200,
            max_chars_total=20000,
            max_chunks=max_chunks,
        )

    # Otherwise use your sentence-based chunker (the one you already have)
    print(f"--using chunk_text len {t_len} punct {punct}")
    chunks = chunk_text(t, max_words=max_words, overlap_words=overlap_words, min_words=min_words)
    return chunks[:max_chunks]

In [ ]:
# load dataset
# text is the text
# label is the class ID: an integer in [0, 19] 
# label_text is the ng name
ds = load_dataset("SetFit/20_newsgroups")

# TODO: Inspect schema
print(ds)
print(ds["train"][0].keys())


In [ ]:
# print(ds["train"][0])
ds["train"][0]

In [ ]:
# TODO: Print Unique label_text values in the dataset


In [ ]:
# Create chunks (keep a subset for speed in class):
MAX_DOCS = 2000          # 800 tune for demo speed
MAX_CHUNKS_PER_DOC = 6  # 6 avoid exploding index size
CHUNK_CFG = dict(max_words=120, overlap_words=30, min_words=30)

train = ds["train"].select(range(min(MAX_DOCS, len(ds["train"]))))
print(f"train len {len(train)}")

t0 = time.perf_counter()

chunk_records: List[Dict[str, Any]] = []
pid = 1  # Qdrant point id (unique)

for doc_id, row in enumerate(train):
    if doc_id % 50 == 0:
        print("doc_id:", doc_id, "points:", len(chunk_records))

    text = row["text"]
    label = row["label"]  # int
    label_text = row.get("label_text", None)  # some versions include this; if not, ignore

    # chunk text using max_words=120, overlap=30, min words=30, max chunks per doc=6
    chunks = chunk_text_hybrid(row["text"],max_words=120,overlap_words=30,min_words=30,max_chunks=MAX_CHUNKS_PER_DOC,)
    if not chunks:
        continue

    print(f"Chunks done {len(chunks)} doc id {doc_id} chunks {chunks}")

    # Limit chunks per doc for predictable runtime
    chunks = chunks[:MAX_CHUNKS_PER_DOC]

    for chunk_id, chunk in enumerate(chunks):
        # print(f"--chunk {chunk_id} (len {len(chunk)}): {chunk}")

        chunk_records.append({
            "id": pid,                 # Qdrant point id
            "doc_id": doc_id,          # original document index in this subset
            "chunk_id": chunk_id,
            "text": chunk,
            "label": int(label),
            "label_text": label_text,
            "source": "SetFit/20_newsgroups",
            "split": "train",
        })
        pid += 1

t1 = time.perf_counter()
print(f"Chunking: docs={len(train)} chunks={len(chunk_records)} time={t1-t0:.3f}s")



In [ ]:
# TODO: print
# - nbr of docs in train
# - nbr of chunks created
# - max chunk length
# - ave chunk length


### About chunk overlap

Typical “tail of previous chunk appears at start of next chunk” pattern.

Concrete examples from the list:  
	•	Chunk 1 → Chunk 2 overlap.  
	•	Chunk 1 ends with: ... "this summer" but haven't heard anymore on it - and since i don't have access to macleak, i was wondering if anybody out there had more info...   
	•	Chunk 2 starts with: make an appearence "this summer" but haven't heard anymore on it - and since i don't have access to macleak, i was wondering if anybody out there had more info...   
	•	That repeated segment is overlap.   
	•	Chunk 2 → Chunk 3 overlap.  
	•	Chunk 2 ends with: ... but i don't really have a feel for how much "better" the display is (yea, it looks great in the store, but is that all "wow" or is it really that good?).  
	•	Chunk 3 starts with: don't really have a feel for how much "better" the display is (yea, it looks great in the store, but is that all "wow" or is it really that good?).   
	•	Same sentence repeated (with only the first word trimmed) ⇒ overlap.   
	•	Chunk 3 → Chunk 4 overlap.  
	•	Chunk 3 ends with: ... figured the opinions of somebody who actually uses the machine daily might prove helpful). * how well does hellcats perform?   
	•	Chunk 4 starts with: around with the machines in a computer store breifly and figured the opinions of somebody who actually uses the machine daily might prove helpful). * how well does hellcats perform? ;).  
	•	Again, repeated tail ⇒ overlap.   

So yes: your chunker is producing overlap (likely via overlap_chars or overlap logic that prepends a tail to the next chunk).

One note: your overlap here looks a bit too large / “mid-sentence cut” at chunk starts (“make an appearence…”, “don’t really…”). That’s normal with character overlap, but if you want cleaner starts you can:  
	•	reduce overlap_chars (e.g., 200 → 100), or  
	•	align overlap to whitespace boundaries (trim to the first space), or  
	•	use word-based overlap rather than char-based overlap.   

In [ ]:
# embed chunks
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

t0 = time.perf_counter()

texts = [r["text"] for r in chunk_records]
emb = model.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True)

t1 = time.perf_counter()

print(f"Embeddings: {emb.shape} took {t1-t0:.3f}s", )  # (N, 384)
VECTOR_SIZE = emb.shape[1]

In [ ]:
# TODO: Create Qdrant collection + upsert
QDRANT_URL = "http://localhost:6333"
COLLECTION = "newsgroups_chunks"

# TODO: Create client
client = ...

# TODO: delete any existing collection

# TODO: Create collection using COSINE distance

# TODO: Build points structure
points = []
for r, v in ...

# TODO: Batch upsert (important for speed) using size 512 
BATCH = 512
t0 = time.perf_counter()
...

t1 = time.perf_counter()

print(f"Upsert: points={len(points)} time={t1-t0:.3f}s")

In [ ]:
# Query helpers
def query_similar(query: str, k: int = 5, topic: str | None = None):
    qvec = model.encode([query], normalize_embeddings=True)[0]

    # TODO: comment this call
    qfilter = None
    if topic is not None:
        qfilter = Filter(must=[FieldCondition(key="label_text", match=MatchValue(value=topic))])

    # TODO: comment this call
    res = client.query_points(
        collection_name=COLLECTION,
        query=qvec.tolist(),
        limit=k,
        query_filter=qfilter,
        with_payload=True,
        with_vectors=False,
    )
    return res.points

def print_hits(hits, max_chars=280):
    for h in hits:
        p = h.payload
        print(f"\nscore={h.score:.4f}  topic={p['label_text']}  doc={p['doc_id']} chunk={p['chunk_id']}")
        print(p["text"][:max_chars].replace("\n", " "), "...")


In [ ]:
# TODO: make a query to find out closest text to 
# - "PC graphics cards, OpenGL, drivers and 3D rendering on Windows"
# - print 5 most closest replies and their similarities, topics and doc id

# TODO: make a filtered query to find out closest text to from topic 'comp.graphics'
# - "drivers and rendering pipeline"
# - print 5 most closest replies and their similarities, topics and doc id


In [ ]:
# TODO: make a query to find out closest text to 
# - "2-door sports car, from the late 60s early 70s"
# - print 5 most closest replies and their similarities, topics and doc id

# TODO: make a filtered query to find out closest text to from topic 'rec.autos'
# - "2-door sports car, from the late 60s early 70s"
# - print 5 most closest replies and their similarities, topics and doc id

